In [15]:
from pathlib import Path
from dataclasses import dataclass, field, asdict
import numpy as np
import os

from torch.utils.data import Dataset, DataLoader
import torch

from utils import load_config, pad_collate, get_ntrainparams, prepare_output, save_results
import utaeT

In [16]:
class PredTreeDataset(Dataset):
    # Data set with only image patches and dates for prediction

    def __init__(self,patches,dates):
        self.patches = patches
        self.dates = dates

    def __len__(self):
        return len(self.patches)
    
    def __getitem__(self, idx):
        x = self.patches[idx]
        date = self.dates[idx]

        x = torch.tensor(x, dtype=torch.float32)
        date = torch.tensor(date, dtype=torch.float32)

        return x, date

In [ ]:
@dataclass
class ConfigPredPaths:
    # Dataset / paths
    config_path: Path = field(default_factory=lambda: Path.cwd() / "results_train" / "conf.json")

    model_dir: Path = field(default_factory=lambda: Path.cwd() / "results_train")
    res_dir: Path = field(default_factory=lambda: Path.cwd() / "results_pred" / "warsaw")
    imp_dir: Path = Path.cwd().parents[0] / "data" / "preprocessed prediction data"
    paths_sentinel_data: list = None
    paths_date_data: list = None
    def __post_init__(self):
        self.paths_sentinel_data = [
            #self.imp_dir / "satellite_patches_ath.npy",
            #self.imp_dir / "satellite_patches_bud.npy",
            #self.imp_dir / "satellite_patches_mil.npy",
            #self.imp_dir / "satellite_patches_pra.npy",
            self.imp_dir / "satellite_patches_war.npy",
        ]
        self.paths_date_data = [
            #self.imp_dir / "date_patches_ath.npy",
            #self.imp_dir / "date_patches_bud.npy",
            #self.imp_dir / "date_patches_mil.npy",
            #self.imp_dir / "date_patches_pra.npy",
            self.imp_dir / "date_patches_war.npy",
        ]

In [18]:
@dataclass
class ConfigPred:
    rdm_seed = 42

    # computation and display
    display_step: int = 1
    num_workers: int = 4
    batch_size: int = 4
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    def torch_device(self):
        return torch.device(self.device)

    fold: int = 1

In [19]:
def predict(model, data_loader, config_train, config_pred):

    # set model in eval mode
    model.eval()

    all_preds = []
    all_probs = []

    with torch.no_grad():
        
        # go through each batch in the data
        for i, batch in enumerate(data_loader):

            x, dates = batch

            x = x.to(config_pred.device)
            dates = dates.to(config_pred.device)

            # forward pass
            presence_logits, lambda_pred = model(x, batch_positions=dates)

            # probabilities
            presence_prob = torch.sigmoid(presence_logits)

            # final count prediction
            pred_count = ((presence_prob > config_train.presence_threshold).float() * torch.expm1(lambda_pred))

            all_preds.append(pred_count.cpu().numpy())
            all_probs.append(presence_prob.cpu().numpy())

            if (i + 1) % config_pred.display_step == 0:
                print(f"Step [{i + 1}/{len(data_loader)}]")

    return all_preds, all_probs

In [20]:
def main(config_train, config_pred,config_pred_paths):
    # set random seed and device
    np.random.seed(config_pred.rdm_seed)
    torch.manual_seed(config_pred.rdm_seed)
    device = torch.device(config_pred.device)

    sentinel_list = []
    date_list = []
    
    for p_s, p_d in zip(config_pred_paths.paths_sentinel_data,config_pred_paths.paths_date_data):
        sentinel_list.append(np.load(p_s, mmap_mode='r'))
        date_list.append(np.load(p_d, mmap_mode='r'))

    # merge all cities
    pred_x = np.concatenate(sentinel_list, axis=0)
    pred_d = np.concatenate(date_list, axis=0)

    dt_pred  = PredTreeDataset(pred_x, pred_d)

    # create dataloaders using padding
    collate_fn = lambda x: pad_collate(x, pad_value=config_train.pad_value)
    pred_loader = DataLoader(
        dt_pred,
        batch_size=config_pred.batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=collate_fn,
    )

    print(f"pred {len(dt_pred)}")

    # load model structure
    model = utaeT.UTAE(config_train)
    model = model.to(device)

    config_pred.N_params = get_ntrainparams(model)
    print(model)
    print("TOTAL TRAINABLE PARAMETERS :", config_pred.N_params)

    # create lists for pred evaluation
    all_preds_folds = []
    all_targets_folds = []

    # define which folds should be predicted
    if config_pred.fold is not None:
        folds_to_run = [config_pred.fold - 1]
    else:
        folds_to_run = range(5)

    # prepare output folders
    prepare_output(config_pred_paths,config_pred)

    for fold in folds_to_run:
        # Load weights
        sd = torch.load(
            os.path.join(config_pred_paths.model_dir, "Fold_{}".format(fold+1), "model.pth.tar"),
            map_location=device,
        )
        model.load_state_dict(sd["state_dict"])
    
        print(f"\n=== Running Fold {fold+1} ===")
        
        # Predicting
        print("Predicting . . .")
        model.eval()
        preds, probs = predict(model, pred_loader, config_train, config_pred)

        all_preds_folds.extend(preds)
        save_results(fold=fold + 1, config=config_pred,config_paths=config_pred_paths,preds=preds, probs=probs)

In [21]:
if __name__ == "__main__":
    config_pred = ConfigPred()
    config_pred_paths = ConfigPredPaths()
    config_train = load_config(config_pred_paths)

    main(config_train,config_pred,config_pred_paths)

pred 770
UTAE(
  (in_conv): ConvBlock(
    (conv): ConvLayer(
      (conv): Sequential(
        (0): Conv2d(5, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (1): GroupNorm(8, 32, eps=1e-05, affine=True)
        (2): ReLU()
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (4): GroupNorm(8, 32, eps=1e-05, affine=True)
        (5): ReLU()
      )
    )
  )
  (down_blocks): ModuleList(
    (0): DownConvBlock(
      (down): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(8, 32, eps=1e-05, affine=True)
          (2): ReLU()
        )
      )
      (conv1): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(8, 32, eps=1e-05, affine=True)
          (2): ReLU()
     